In [ ]:
"""Today is to do with Chain-Of-Thought Logging(seeing the agent's brain)
Trace object: Designing a reactTrace dataclass that stores every step, number, timestamp,
raw model output, parsed action, action input, observation and whether the step was a final
answer or an error recovery.
Logger intergration: Wrap my day 8 run_react_loop function so every loop iteratrion appends 
to a ReactTrace instead of just printing. The console output should still be human-readable,
but the trace object is the real output.
Replay function: Write a replay_trace(trace) function that pretty-prints any saved trace in
a clean, readable format. this is how I'll review and debug agent runs without re-executing them.
Export: Save traces to JSON so I can load them later in Month 3 when you're running RAGAS evaluation.

"""

import os
import re
import json
from groq import Groq
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext

load_dotenv(find_dotenv()) 

Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)

client = Groq()

@dataclass
class TraceStep:
    step_number: int
    timestamp: str
    raw_llm_output: str
    parsed_action: Optional[str]
    parsed_action_input: Optional[str]
    observation: Optional[str]
    is_final_answer: bool
    is_error_recovery: bool

@dataclass
class ReActTrace:
    question: str
    final_answer: Optional[str]
    total_steps: int
    success: bool
    started_at: str
    finished_at: str
    steps: list[TraceStep] = field(default_factory= list)

    def to_json(self, filepath: str):

        """# Ensuring the parent directory (e.g.,  'traces/') exists before writing
        os.makedirs(os.path.dirname(filepath), exist_ok= True)"""

        with open(filepath, "w") as f:
            json.dump(asdict(self), f, indent = 2)
        print(f"👍 Trace saved to {filepath}")

    @classmethod
    def from_json(cls, filepath: str):
        with open(filepath) as f:
            data = json.load(f)
        steps = [TraceStep(**s) for s in data.pop("steps")]
        trace = cls(**data)
        trace.steps = steps
        return trace

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# first connecting to existing vector store
# connecting to existing vectorstore
# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)

# creating the primary retrieval tool
'''apple_10k_expert = QueryEngineTool(
    query_engine= index.as_query_engine(similarity_top_k = 15),
    metadata= ToolMetadata(
        name = 'apple_10k_expert',
        description= "Search Through Apple's 10-K filings for historical financial data, revenue figures, and risk factors."
    )

)'''

print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [10]:
# Tools and parser from day 8


def search_local_docs(query:str) -> str:
    """Searches my local Knowledge base containing historical Apple 10-K financial documents
    (covering fiscal years up to 2024). Use this to retrieve historical sales, net revenue,
    and internal corporate performance figures.
    
    CRITICAL: Do not pass comparative or converstional questions here.
    Convert queries into strict financial line items, such as:
    - 'Apple consolidated statements of operations net sales' 
    - 'Apple total net slaes 2023 -2024' 
    - 'Summary of operations data'
    """

    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)
    return "\n\n".join([doc.node.get_content() for doc in results])

def get_doc_years(_: str = "") -> str:
    """Returns the list of available years in the 10k pdf"""
    return "Available years: 2022, 2023, 2024"

def calculator(expression: str) -> str:
    """Evaluates a basic math expression"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"
    
TOOLS = {
    "search_local_docs": search_local_docs,
    "get_doc_years": get_doc_years,
    "calculator": calculator,
}

REACT_SYSTEM_PROMPT = """You are a careful search assistant that solves problems step by step

You have access to these tools:
-search_local_docs(query): searches the user's local pdf documents.
-get_doc_years(): lists the number of years available in the apple 10K document.
-calculator(expression): evaluates a math expression. CRITICAL: Only use this tool if the user's question explicitly requires a mathematical calculation. DO NOT use it to add up calendar years or calculate unprompted statistics from the data table.

Use EXACTLY this format, one step at a time:

Thought: <your reasoning about what to do next>
Action: <one of: search_local_docs, get_doc_years, calculator>
Action Input: <the input to the tool>

CRITICAL EXIT CONDITION: After you receive an Observation, if that observation contains the direct answer to the user's question, you must STOP making tool calls and output your Final Answer. Do not perform extra unnecessary steps.


When you are ready to give your final answe, use this exact format:
Thought: <final reasoning showing you have all the information needed>
Final Answer: <your clear, direct answer to the user>

Never skip the Thought step. Never output Action and Final Answer in the same turn.


"""

# below is the code for the parser

def parse_action(text: str):
    """Pulls Action and Action Input out of the model's raw text output"""
    action_match = re.search(r"Action:\s*(\w+)", text)
    input_match = re.search(r"Action Input:\s*(.+)", text)
    final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)

    if final_match:
        return None, None, final_match.group(1).strip()
    
    action = action_match.group(1).strip() if action_match else None
    action_input = input_match.group(1).strip() if input_match else None
    return action, action_input, None


In [11]:
# now we write the code for the logged react loop

def run_logged_react_loop(question: str, max_steps: int = 10) -> ReActTrace:
    conversation = f"{REACT_SYSTEM_PROMPT}\n\nQuestion: {question}\n"
    started_at = datetime.now().isoformat()

    trace = ReActTrace(
        question= question,
        final_answer= None,
        total_steps= 0,
        success= False,
        started_at= started_at,
        finished_at="",
    )
    
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}\n")

    for step_num in range(1, max_steps + 1):
        timestamp = datetime.now().isoformat()

        response = client.chat.completions.create(
            model = "llama-3.3-70b-versatile",
            messages = [{"role": "user", "content": conversation}],
            temperature= 0,
            stop =["Observation:"]
        )
        raw_output = response.choices[0].message.content

        print(f"--- Step {step_num} ----")
        print(raw_output)

        action, action_input, final_answer = parse_action(raw_output)
        is_final = final_answer is not None
        is_error_recovery = False
        observation = None

        if is_final:
            trace.steps.append(TraceStep(
                step_number= step_num,
                timestamp= timestamp,
                raw_llm_output= raw_output,
                parsed_action= None,
                parsed_action_input= None,
                observation= None,
                is_final_answer= True,
                is_error_recovery= False,
            ))
            trace.final_answer = final_answer
            trace.success = True
            print(f"\n Final Answer: {final_answer}")
            break 

        if action and action in TOOLS:
            observation = TOOLS[action](action_input)
        elif action:
            observation = (
                f"Error: tool '{action}' does not exist. "
                f"Available tools: {list(TOOLS.keys())}"
            )
            is_error_recovery = True
        else:
            observation = "Error: could not parse and Action. Follow the Thought/Action/Action Input format exactly."
            is_error_recovery = True

        print(f"Observation: {observation}\n")
        conversation += f"{raw_output}\nObservation: {observation}"

        trace.steps.append(TraceStep(
            step_number= step_num,
            timestamp= timestamp,
            raw_llm_output= raw_output,
            parsed_action= action,
            parsed_action_input= action_input,
            observation= observation,
            is_final_answer= False,
            is_error_recovery= is_error_recovery,

        ))
    
    trace.total_steps = len(trace.steps)
    trace.finished_at = datetime.now().isoformat()

    if not trace.success:
        print("🤥 Max steps reached without Final Answer.")
    return trace



In [12]:
# the replay Function
def replay_trace(trace: ReActTrace):
    print(f"\n{'='*60}")
    print(f"TRACE REPLAY")
    print(f"Question: {trace.question}")
    print(f"Success: {trace.success}")
    print(f"Steps: {trace.total_steps}")
    print(f"Started : {trace.started_at}")
    print(f"Finished: {trace.finished_at}")
    print(f"{'='*60}")


    for step in trace.steps:
        print(f"\n[step {step.step_number}] {step.timestamp}")
        if step.is_error_recovery:
            print("🫤 ERROE RECOVERY STEP")
        print(step.raw_llm_output)
        if step.observation:
            print(f"Observation: {step.observation}")
        if step.is_final_answer:
            print(f"\n👍 Final Answer: {trace.final_answer}")
    print(f"\n{'='*60}\n")

In [13]:
# RUN, SAVE and REPLAY

questions = [
    "what years are available in my docs?",
    "Search my docs for anything abour apple sales and summarize it.",
    "What is 3456 divided by 12, and then multiplied by 7?",
]


all_traces = []

# creating folder here
os.makedirs("traces", exist_ok= True)

for q in questions:
    trace = run_logged_react_loop(q)
    filepath = f"traces/day9_trace_{len(all_traces) + 1}.json"
    trace.to_json(filepath= filepath)
    all_traces.append(trace)


# Replay any saved trace from disk
loaded = ReActTrace.from_json("traces/day9_trace_1.json")


# Quick summary across all rune
print("\n⚡Run Summary")
print(f"{'Question':<55} {'Steps':>5} {'Success':>8}")
print("-" * 70)
for t in all_traces:
    print(f"{t.question[:54]:<55} {t.total_steps:>5} {str(t.success):>8}")



Question: what years are available in my docs?

--- Step 1 ----
Thought: The user is asking for the years available in their local documents, but the tool that lists years is specifically for the Apple 10K document. However, there's also a search function for local documents. Since the question is about the user's local docs, I should use the search function to find the relevant information.

Action: search_local_docs
Action Input: years in local documents
Observation: Unless otherwise stated, references to particular years, quarters, months and periods refer to the Company’s fiscal years ended in September and the associated quarters, months and periods of those fiscal years.



These leases typically have original terms not exceeding 10 years and generally contain multiyear renewal options, some of which are reasonably certain of exercise.



Unless otherwise stated, all information presented herein is based on the Company’s fiscal calendar, and references to particular years, quart